> **Run:** Clean confirmation
>
> **Evidence label:** DEVELOPMENT / PRELIMINARY
>
> **Original file:** `text2sql_execution_outcome_confirmation_repilot.ipynb`
>
> **Purpose:** This notebook tests whether that effect survives a cleaner design.
>
> **Drive output folder:** `Text2SQL_ExecutionOutcome_Confirmation`
>
> This notebook is not part of the confirmatory protocol.

# Confirmation Re-Pilot — Execution-Outcome Sensitivity in Text-to-SQL Verification

## 1. Goal

The first pilot showed a very large increase in rejection of **correct SQL** when the same candidate was paired with an `EMPTY` execution outcome.

This notebook tests whether that effect survives a cleaner design.

### What changed from the first pilot

1. **Verdict only** — no SQL revision is requested.
2. **Four feedback conditions**:
   - `NONE`
   - `SUCCESS_ONLY`
   - `EMPTY`
   - `NONEMPTY`
3. The primary comparison is now:
   - **EMPTY vs SUCCESS_ONLY**

   This isolates zero-row cardinality from the mere fact that execution succeeded.
4. Only a **conservatively pre-screened set of correct SQL candidates** is used.
5. No wrong-SQL mutation is needed for this confirmation re-pilot.
6. Every item is paired: the **question, schema, and candidate SQL are identical** across all four conditions.

### Main estimand

```text
Empty rejection shift
=
P(reject correct SQL | EMPTY)
-
P(reject correct SQL | SUCCESS_ONLY)
```

A strong positive value means that adding only `Returned row count: 0` makes the verifier more likely to reject an otherwise identical correct SQL candidate.

### Default size

- 40 pre-screened correct SQL candidates
- 4 conditions
- 1 model by default (`Qwen/Qwen3-8B`)
- **160 model calls**

A second model can be enabled after the Qwen readout.

In [ ]:
# ============================================================
# 0. GOOGLE DRIVE + PERSISTENT WORKDIR
# ============================================================
import os, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

    DRIVE_ROOT = Path("/content/drive/MyDrive")
    WORK = DRIVE_ROOT / "Text2SQL_ExecutionOutcome_Confirmation"
    print("✓ Google Drive mounted")
else:
    WORK = Path("./Text2SQL_ExecutionOutcome_Confirmation")
    print("Not running in Colab; using local WORK directory.")

WORK.mkdir(parents=True, exist_ok=True)

# Keep large model cache on local ephemeral storage for speed.
if IN_COLAB:
    os.environ.setdefault("HF_HOME", "/content/hf_cache")

print("Persistent WORK directory:")
print(WORK.resolve())

In [ ]:
# ============================================================
# 1. AUTOSETUP
# ============================================================
import subprocess, sys

REQ = [
    "transformers>=4.51.0,<5.0",
    "accelerate>=1.2.0",
    "bitsandbytes>=0.45.0",
    "huggingface_hub>=0.27.0",
    "pandas>=2.0.0",
    "numpy>=1.26.0,<2.4",
    "scipy>=1.11.0",
    "tqdm>=4.66.0",
    "sentencepiece>=0.2.0",
]

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "--upgrade-strategy", "only-if-needed",
    *REQ
])

import json, random, time, re, gc
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from scipy.stats import binomtest
import transformers

print("=" * 72)
print("Python       :", sys.version.split()[0])
print("Torch        :", torch.__version__)
print("Transformers :", transformers.__version__)
print("CUDA         :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU          :", torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print("VRAM         :", round(props.total_memory / 1024**3, 1), "GB")
else:
    print("WARNING: no CUDA GPU detected.")

print("=" * 72)

In [ ]:
# ============================================================
# 2. CONFIG
# ============================================================
CFG = {
    "seed": 42,

    # One model first. Add Mistral only if the Qwen confirmation is informative.
    "models": [
        "Qwen/Qwen3-8B",
        # "mistralai/Mistral-7B-Instruct-v0.3",
    ],

    # Inference
    "load_in_4bit": True,
    "temperature": 0.0,
    "max_new_tokens": 48,
    "trust_remote_code": False,

    # Trial order / resume
    "shuffle_trials": True,
    "overwrite_results": False,

    # Audit policy
    # The embedded 40-item set has already been conservatively pre-screened.
    # For a paper-quality run, the team should fill pilot_manual_audit.csv
    # independently and rerun with require_team_audit=True.
    "require_team_audit": False,

    # Internal go/no-go interpretation only; not a formal significance threshold.
    "strong_go_pp": 15.0,
    "moderate_go_pp": 10.0,
}

random.seed(CFG["seed"])
np.random.seed(CFG["seed"])

HF_TOKEN = (
    os.environ.get("HF_TOKEN")
    or os.environ.get("HUGGINGFACE_TOKEN")
    or os.environ.get("HUGGING_FACE_HUB_TOKEN")
)

print("WORK:", WORK)
print("HF token detected:", bool(HF_TOKEN))

## 2. Candidate-set provenance

The notebook embeds 40 correct-SQL candidates from the previous 50-item pilot.

The re-pilot intentionally **does not reuse all 50**. Ten items were removed before seeing any result from this confirmation experiment because they were judged unsuitable for a clean correct-SQL causal test, including cases with:

- ambiguous natural-language semantics;
- questionable benchmark gold semantics;
- near-duplicate/paraphrased tasks;
- wording corruption;
- or other avoidable validity concerns.

This is still a **confirmation pilot**, not the final annotation protocol. For a paper-quality full study, use independent human annotators and adjudication.

The embedded set makes this notebook self-contained: it does not need Spider/BIRD downloads to run the verifier experiment.

In [ ]:
# ============================================================
# 3. EMBEDDED PRE-SCREENED CORRECT-SQL SET
# ============================================================
CURATED_DATA_JSON = r"""[{"item_id": "cre_Doc_Template_Mgt__000", "db_id": "cre_Doc_Template_Mgt", "question": "What is the document name and template id for document with description with the letter 'w' in it?", "schema": "TABLE Documents (Document_ID INTEGER PRIMARY KEY, Template_ID INTEGER, Document_Name VARCHAR(255), Document_Description VARCHAR(255), Other_Details VARCHAR(255));\nFOREIGN KEY Documents.Template_ID -> Templates.Template_ID\nTABLE Paragraphs (Paragraph_ID INTEGER PRIMARY KEY, Document_ID INTEGER, Paragraph_Text VARCHAR(255), Other_Details VARCHAR(255));\nFOREIGN KEY Paragraphs.Document_ID -> Documents.Document_ID\nTABLE Ref_Template_Types (Template_Type_Code CHAR(15) PRIMARY KEY, Template_Type_Description VARCHAR(255));\nTABLE Templates (Template_ID INTEGER PRIMARY KEY, Version_Number INTEGER, Template_Type_Code CHAR(15), Date_Effective_From DATETIME, Date_Effective_To DATETIME, Template_Details VARCHAR(255));\nFOREIGN KEY Templates.Template_Type_Code -> Ref_Template_Types.Template_Type_Code", "candidate_sql": "SELECT document_name ,  template_id FROM Documents WHERE Document_Description LIKE \"%w%\"", "source_result_n": 2, "source_actual_empty": 0, "pre_screened_correct": true, "pre_screen_note": "Retained after conservative manual review of the original 50-item pilot set."}, {"item_id": "world_1__002", "db_id": "world_1", "question": "Which language is spoken by the largest number of countries?", "schema": "TABLE city (ID INTEGER PRIMARY KEY, Name char(35), CountryCode char(3), District char(20), Population INTEGER);\nFOREIGN KEY city.CountryCode -> country.Code\nTABLE country (Code char(3) PRIMARY KEY, Name char(52), Continent TEXT, Region char(26), SurfaceArea float(10,2), IndepYear INTEGER, Population INTEGER, LifeExpectancy float(3,1), GNP float(10,2), GNPOld float(10,2), LocalName char(45), GovernmentForm char(45), HeadOfState char(60), Capital INTEGER, Code2 char(2));\nTABLE countrylanguage (CountryCode char(3) PRIMARY KEY, Language char(30) PRIMARY KEY, IsOfficial TEXT, Percentage float(4,1));\nFOREIGN KEY countrylanguage.CountryCode -> country.Code", "candidate_sql": "SELECT LANGUAGE FROM countrylanguage GROUP BY LANGUAGE ORDER BY count(*) DESC LIMIT 1", "source_result_n": 1, "source_actual_empty": 0, "pre_screened_correct": true, "pre_screen_note": "Retained after conservative manual review of the original 50-item pilot set."}, {"item_id": "singer__003", "db_id": "singer", "question": "What are the names of the singers who are not French citizens?", "schema": "TABLE singer (Singer_ID INT PRIMARY KEY, Name TEXT, Birth_Year REAL, Net_Worth_Millions REAL, Citizenship TEXT);\nTABLE song (Song_ID INT PRIMARY KEY, Title TEXT, Singer_ID INT, Sales REAL, Highest_Position REAL);\nFOREIGN KEY song.Singer_ID -> singer.Singer_ID", "candidate_sql": "SELECT Name FROM singer WHERE Citizenship != \"France\"", "source_result_n": 7, "source_actual_empty": 0, "pre_screened_correct": true, "pre_screen_note": "Retained after conservative manual review of the original 50-item pilot set."}, {"item_id": "tvshow__005", "db_id": "tvshow", "question": "List top 3 highest Rating  TV series. List the TV series's Episode and Rating.", "schema": "TABLE Cartoon (id REAL PRIMARY KEY, Title TEXT, Directed_by TEXT, Written_by TEXT, Original_air_date TEXT, Production_code REAL, Channel TEXT);\nFOREIGN KEY Cartoon.Channel -> TV_Channel.id\nTABLE TV_Channel (id TEXT PRIMARY KEY, series_name TEXT, Country TEXT, Language TEXT, Content TEXT, Pixel_aspect_ratio_PAR TEXT, Hight_definition_TV TEXT, Pay_per_view_PPV TEXT, Package_Option TEXT);\nTABLE TV_series (id REAL PRIMARY KEY, Episode TEXT, Air_Date TEXT, Rating TEXT, Share REAL, 18_49_Rating_Share TEXT, Viewers_m TEXT, Weekly_Rank REAL, Channel TEXT);\nFOREIGN KEY TV_series.Channel -> TV_Channel.id", "candidate_sql": "SELECT Episode ,  Rating FROM TV_series ORDER BY Rating DESC LIMIT 3;", "source_result_n": 3, "source_actual_empty": 0, "pre_screened_correct": true, "pre_screen_note": "Retained after conservative manual review of the original 50-item pilot set."}, {"item_id": "flight_2__006", "db_id": "flight_2", "question": "What are the airline names and abbreviations for airlines in the USA?", "schema": "TABLE airlines (uid INTEGER PRIMARY KEY, Airline TEXT, Abbreviation TEXT, Country TEXT);\nTABLE airports (City TEXT, AirportCode TEXT PRIMARY KEY, AirportName TEXT, Country TEXT, CountryAbbrev TEXT);\nTABLE flights (Airline INTEGER PRIMARY KEY, FlightNo INTEGER PRIMARY KEY, SourceAirport TEXT, DestAirport TEXT);\nFOREIGN KEY flights.DestAirport -> airports.AirportCode\nFOREIGN KEY flights.SourceAirport -> airports.AirportCode", "candidate_sql": "SELECT Airline ,  Abbreviation FROM AIRLINES WHERE Country  =  \"USA\"", "source_result_n": 12, "source_actual_empty": 0, "pre_screened_correct": true, "pre_screen_note": "Retained after conservative manual review of the original 50-item pilot set."}, {"item_id": "cre_Doc_Template_Mgt__007", "db_id": "cre_Doc_Template_Mgt", "question": "What is the id and type code for the template used by the most documents?", "schema": "TABLE Documents (Document_ID INTEGER PRIMARY KEY, Template_ID INTEGER, Document_Name VARCHAR(255), Document_Description VARCHAR(255), Other_Details VARCHAR(255));\nFOREIGN KEY Documents.Template_ID -> Templates.Template_ID\nTABLE Paragraphs (Paragraph_ID INTEGER PRIMARY KEY, Document_ID INTEGER, Paragraph_Text VARCHAR(255), Other_Details VARCHAR(255));\nFOREIGN KEY Paragraphs.Document_ID -> Documents.Document_ID\nTABLE Ref_Template_Types (Template_Type_Code CHAR(15) PRIMARY KEY, Template_Type_Description VARCHAR(255));\nTABLE Templates (Template_ID INTEGER PRIMARY KEY, Version_Number INTEGER, Template_Type_Code CHAR(15), Date_Effective_From DATETIME, Date_Effective_To DATETIME, Template_Details VARCHAR(255));\nFOREIGN KEY Templates.Template_Type_Code -> Ref_Template_Types.Template_Type_Code", "candidate_sql": "SELECT T1.template_id ,  T2.Template_Type_Code FROM Documents AS T1 JOIN Templates AS T2 ON T1.template_id  =  T2.template_id GROUP BY T1.template_id ORDER BY count(*) DESC LIMIT 1", "source_result_n": 1, "source_actual_empty": 0, "pre_screened_correct": true, "pre_screen_note": "Retained after conservative manual review of the original 50-item pilot set."}, {"item_id": "cre_Doc_Template_Mgt__008", "db_id": "cre_Doc_Template_Mgt", "question": "What is the version number and template type code for the template with version number later than 5?", "schema": "TABLE Documents (Document_ID INTEGER PRIMARY KEY, Template_ID INTEGER, Document_Name VARCHAR(255), Document_Description VARCHAR(255), Other_Details VARCHAR(255));\nFOREIGN KEY Documents.Template_ID -> Templates.Template_ID\nTABLE Paragraphs (Paragraph_ID INTEGER PRIMARY KEY, Document_ID INTEGER, Paragraph_Text VARCHAR(255), Other_Details VARCHAR(255));\nFOREIGN KEY Paragraphs.Document_ID -> Documents.Document_ID\nTABLE Ref_Template_Types (Template_Type_Code CHAR(15) PRIMARY KEY, Template_Type_Description VARCHAR(255));\nTABLE Templates (Template_ID INTEGER PRIMARY KEY, Version_Number INTEGER, Template_Type_Code CHAR(15), Date_Effective_From DATETIME, Date_Effective_To DATETIME, Template_Details VARCHAR(255));\nFOREIGN KEY Templates.Template_Type_Code -> Ref_Template_Types.Template_Type_Code", "candidate_sql": "SELECT version_number ,  template_type_code FROM Templates WHERE version_number  >  5", "source_result_n": 9, "source_actual_empty": 0, "pre_screened_correct": true, "pre_screen_note": "Retained after conservative manual review of the original 50-item pilot set."}, {"item_id": "flight_2__009", "db_id": "flight_2", "question": "Give the city and country for the Alton airport.", "schema": "TABLE airlines (uid INTEGER PRIMARY KEY, Airline TEXT, Abbreviation TEXT, Country TEXT);\nTABLE airports (City TEXT, AirportCode TEXT PRIMARY KEY, AirportName TEXT, Country TEXT, CountryAbbrev TEXT);\nTABLE flights (Airline INTEGER PRIMARY KEY, FlightNo INTEGER PRIMARY KEY, SourceAirport TEXT, DestAirport TEXT);\nFOREIGN KEY flights.DestAirport -> airports.AirportCode\nFOREIGN KEY flights.SourceAirport -> airports.AirportCode", "candidate_sql": "SELECT City ,  Country FROM AIRPORTS WHERE AirportName  =  \"Alton\"", "source_result_n": 0, "source_actual_empty": 1, "pre_screened_correct": true, "pre_screen_note": "Retained after conservative manual review of the original 50-item pilot set."}, {"item_id": "world_1__010", "db_id": "world_1", "question": "What are the population, name and leader of the country with the largest area?", "schema": "TABLE city (ID INTEGER PRIMARY KEY, Name char(35), CountryCode char(3), District char(20), Population INTEGER);\nFOREIGN KEY city.CountryCode -> country.Code\nTABLE country (Code char(3) PRIMARY KEY, Name char(52), Continent TEXT, Region char(26), SurfaceArea float(10,2), IndepYear INTEGER, Population INTEGER, LifeExpectancy float(3,1), GNP float(10,2), GNPOld float(10,2), LocalName char(45), GovernmentForm char(45), HeadOfState char(60), Capital INTEGER, Code2 char(2));\nTABLE countrylanguage (CountryCode char(3) PRIMARY KEY, Language char(30) PRIMARY KEY, IsOfficial TEXT, Percentage float(4,1));\nFOREIGN KEY countrylanguage.CountryCode -> country.Code", "candidate_sql": "SELECT Name ,  population ,  HeadOfState FROM country ORDER BY SurfaceArea DESC LIMIT 1", "source_result_n": 1, "source_actual_empty": 0, "pre_screened_correct": true, "pre_screen_note": "Retained after conservative manual review of the original 50-item pilot set."}, {"item_id": "tvshow__011", "db_id": "tvshow", "question": "What are the titles of all cartoons directed by Ben Jones or Brandon Vietti?", "schema": "TABLE Cartoon (id REAL PRIMARY KEY, Title TEXT, Directed_by TEXT, Written_by TEXT, Original_air_date TEXT, Production_code REAL, Channel TEXT);\nFOREIGN KEY Cartoon.Channel -> TV_Channel.id\nTABLE TV_Channel (id TEXT PRIMARY KEY, series_name TEXT, Country TEXT, Language TEXT, Content TEXT, Pixel_aspect_ratio_PAR TEXT, Hight_definition_TV TEXT, Pay_per_view_PPV TEXT, Package_Option TEXT);\nTABLE TV_series (id REAL PRIMARY KEY, Episode TEXT, Air_Date TEXT, Rating TEXT, Share REAL, 18_49_Rating_Share TEXT, Viewers_m TEXT, Weekly_Rank REAL, Channel TEXT);\nFOREIGN KEY TV_series.Channel -> TV_Channel.id", "candidate_sql": "SELECT Title FROM Cartoon WHERE Directed_by = \"Ben Jones\" OR Directed_by = \"Brandon Vietti\";", "source_result_n": 8, "source_actual_empty": 0, "pre_screened_correct": true, "pre_screen_note": "Retained after conservative manual review of the original 50-item pilot set."}, {"item_id": "battle_death__012", "db_id": "battle_death", "question": "What are the death and injury situations caused by the ship with tonnage 't'?", "schema": "TABLE battle (id INT PRIMARY KEY, name TEXT, date TEXT, bulgarian_commander TEXT, latin_commander TEXT, result TEXT);\nTABLE death (caused_by_ship_id INT, id INT PRIMARY KEY, note TEXT, killed INT, injured INT);\nFOREIGN KEY death.caused_by_ship_id -> ship.id\nTABLE ship (lost_in_battle INT, id INT PRIMARY KEY, name TEXT, tonnage TEXT, ship_type TEXT, location TEXT, disposition_of_ship TEXT);\nFOREIGN KEY ship.lost_in_battle -> battle.id", "candidate_sql": "SELECT T1.killed ,  T1.injured FROM death AS T1 JOIN ship AS t2 ON T1.caused_by_ship_id  =  T2.id WHERE T2.tonnage  =  't'", "source_result_n": 11, "source_actual_empty": 0, "pre_screened_correct": true, "pre_screen_note": "Retained after conservative manual review of the original 50-item pilot set."}, {"item_id": "cre_Doc_Template_Mgt__015", "db_id": "cre_Doc_Template_Mgt", "question": "What are the ids of documents that have 2 or more paragraphs?", "schema": "TABLE Documents (Document_ID INTEGER PRIMARY KEY, Template_ID INTEGER, Document_Name VARCHAR(255), Document_Description VARCHAR(255), Other_Details VARCHAR(255));\nFOREIGN KEY Documents.Template_ID -> Templates.Template_ID\nTABLE Paragraphs (Paragraph_ID INTEGER PRIMARY KEY, Document_ID INTEGER, Paragraph_Text VARCHAR(255), Other_Details VARCHAR(255));\nFOREIGN KEY Paragraphs.Document_ID -> Documents.Document_ID\nTABLE Ref_Template_Types (Template_Type_Code CHAR(15) PRIMARY KEY, Template_Type_Description VARCHAR(255));\nTABLE Templates (Template_ID INTEGER PRIMARY KEY, Version_Number INTEGER, Template_Type_Code CHAR(15), Date_Effective_From DATETIME, Date_Effective_To DATETIME, Template_Details VARCHAR(255));\nFOREIGN KEY Templates.Template_Type_Code -> Ref_Template_Types.Template_Type_Code", "candidate_sql": "SELECT document_id FROM Paragraphs GROUP BY document_id HAVING count(*)  >=  2", "source_result_n": 4, "source_actual_empty": 0, "pre_screened_correct": true, "pre_screen_note": "Retained after conservative manual review of the original 50-item pilot set."}, {"item_id": "employee_hire_evaluation__016", "db_id": "employee_hire_evaluation", "question": "Find the names of employees who never won any award in the evaluation.", "schema": "TABLE employee (Employee_ID INT PRIMARY KEY, Name TEXT, Age INT, City TEXT);\nTABLE evaluation (Employee_ID TEXT PRIMARY KEY, Year_awarded TEXT PRIMARY KEY, Bonus REAL);\nFOREIGN KEY evaluation.Employee_ID -> employee.Employee_ID\nTABLE hiring (Shop_ID INT, Employee_ID INT PRIMARY KEY, Start_from TEXT, Is_full_time bool);\nFOREIGN KEY hiring.Employee_ID -> employee.Employee_ID\nFOREIGN KEY hiring.Shop_ID -> shop.Shop_ID\nTABLE shop (Shop_ID INT PRIMARY KEY, Name TEXT, Location TEXT, District TEXT, Number_products INT, Manager_name TEXT);", "candidate_sql": "SELECT name FROM employee WHERE Employee_ID NOT IN (SELECT Employee_ID FROM evaluation)", "source_result_n": 5, "source_actual_empty": 0, "pre_screened_correct": true, "pre_screen_note": "Retained after conservative manual review of the original 50-item pilot set."}, {"item_id": "cre_Doc_Template_Mgt__018", "db_id": "cre_Doc_Template_Mgt", "question": "What is the template type descriptions for template type code \"AD\".", "schema": "TABLE Documents (Document_ID INTEGER PRIMARY KEY, Template_ID INTEGER, Document_Name VARCHAR(255), Document_Description VARCHAR(255), Other_Details VARCHAR(255));\nFOREIGN KEY Documents.Template_ID -> Templates.Template_ID\nTABLE Paragraphs (Paragraph_ID INTEGER PRIMARY KEY, Document_ID INTEGER, Paragraph_Text VARCHAR(255), Other_Details VARCHAR(255));\nFOREIGN KEY Paragraphs.Document_ID -> Documents.Document_ID\nTABLE Ref_Template_Types (Template_Type_Code CHAR(15) PRIMARY KEY, Template_Type_Description VARCHAR(255));\nTABLE Templates (Template_ID INTEGER PRIMARY KEY, Version_Number INTEGER, Template_Type_Code CHAR(15), Date_Effective_From DATETIME, Date_Effective_To DATETIME, Template_Details VARCHAR(255));\nFOREIGN KEY Templates.Template_Type_Code -> Ref_Template_Types.Template_Type_Code", "candidate_sql": "SELECT template_type_description FROM Ref_template_types WHERE template_type_code  =  \"AD\"", "source_result_n": 1, "source_actual_empty": 0, "pre_screened_correct": true, "pre_screen_note": "Retained after conservative manual review of the original 50-item pilot set."}, {"item_id": "concert_singer__019", "db_id": "concert_singer", "question": "What is the name and capacity for the stadium with highest average attendance?", "schema": "TABLE concert (concert_ID INT PRIMARY KEY, concert_Name TEXT, Theme TEXT, Stadium_ID TEXT, Year TEXT);\nFOREIGN KEY concert.Stadium_ID -> stadium.Stadium_ID\nTABLE singer (Singer_ID INT PRIMARY KEY, Name TEXT, Country TEXT, Song_Name TEXT, Song_release_year TEXT, Age INT, Is_male bool);\nTABLE singer_in_concert (concert_ID INT PRIMARY KEY, Singer_ID TEXT PRIMARY KEY);\nFOREIGN KEY singer_in_concert.Singer_ID -> singer.Singer_ID\nFOREIGN KEY singer_in_concert.concert_ID -> concert.concert_ID\nTABLE stadium (Stadium_ID INT PRIMARY KEY, Location TEXT, Name TEXT, Capacity INT, Highest INT, Lowest INT, Average INT);", "candidate_sql": "SELECT name ,  capacity FROM stadium ORDER BY average DESC LIMIT 1", "source_result_n": 1, "source_actual_empty": 0, "pre_screened_correct": true, "pre_screen_note": "Retained after conservative manual review of the original 50-item pilot set."}, {"item_id": "world_1__020", "db_id": "world_1", "question": "What are the names and areas of countries with the top 5 largest area?", "schema": "TABLE city (ID INTEGER PRIMARY KEY, Name char(35), CountryCode char(3), District char(20), Population INTEGER);\nFOREIGN KEY city.CountryCode -> country.Code\nTABLE country (Code char(3) PRIMARY KEY, Name char(52), Continent TEXT, Region char(26), SurfaceArea float(10,2), IndepYear INTEGER, Population INTEGER, LifeExpectancy float(3,1), GNP float(10,2), GNPOld float(10,2), LocalName char(45), GovernmentForm char(45), HeadOfState char(60), Capital INTEGER, Code2 char(2));\nTABLE countrylanguage (CountryCode char(3) PRIMARY KEY, Language char(30) PRIMARY KEY, IsOfficial TEXT, Percentage float(4,1));\nFOREIGN KEY countrylanguage.CountryCode -> country.Code", "candidate_sql": "SELECT Name ,  SurfaceArea FROM country ORDER BY SurfaceArea DESC LIMIT 5", "source_result_n": 5, "source_actual_empty": 0, "pre_screened_correct": true, "pre_screen_note": "Retained after conservative manual review of the original 50-item pilot set."}, {"item_id": "world_1__022", "db_id": "world_1", "question": "Return the country name and the numbers of languages spoken for each country that speaks at least 3 languages.", "schema": "TABLE city (ID INTEGER PRIMARY KEY, Name char(35), CountryCode char(3), District char(20), Population INTEGER);\nFOREIGN KEY city.CountryCode -> country.Code\nTABLE country (Code char(3) PRIMARY KEY, Name char(52), Continent TEXT, Region char(26), SurfaceArea float(10,2), IndepYear INTEGER, Population INTEGER, LifeExpectancy float(3,1), GNP float(10,2), GNPOld float(10,2), LocalName char(45), GovernmentForm char(45), HeadOfState char(60), Capital INTEGER, Code2 char(2));\nTABLE countrylanguage (CountryCode char(3) PRIMARY KEY, Language char(30) PRIMARY KEY, IsOfficial TEXT, Percentage float(4,1));\nFOREIGN KEY countrylanguage.CountryCode -> country.Code", "candidate_sql": "SELECT COUNT(T2.Language) ,  T1.Name FROM country AS T1 JOIN countrylanguage AS T2 ON T1.Code  =  T2.CountryCode GROUP BY T1.Name HAVING COUNT(*)  >  2", "source_result_n": 149, "source_actual_empty": 0, "pre_screened_correct": true, "pre_screen_note": "Retained after conservative manual review of the original 50-item pilot set."}, {"item_id": "employee_hire_evaluation__023", "db_id": "employee_hire_evaluation", "question": "Find the number of employees hired in each shop; show the shop name as well.", "schema": "TABLE employee (Employee_ID INT PRIMARY KEY, Name TEXT, Age INT, City TEXT);\nTABLE evaluation (Employee_ID TEXT PRIMARY KEY, Year_awarded TEXT PRIMARY KEY, Bonus REAL);\nFOREIGN KEY evaluation.Employee_ID -> employee.Employee_ID\nTABLE hiring (Shop_ID INT, Employee_ID INT PRIMARY KEY, Start_from TEXT, Is_full_time bool);\nFOREIGN KEY hiring.Employee_ID -> employee.Employee_ID\nFOREIGN KEY hiring.Shop_ID -> shop.Shop_ID\nTABLE shop (Shop_ID INT PRIMARY KEY, Name TEXT, Location TEXT, District TEXT, Number_products INT, Manager_name TEXT);", "candidate_sql": "SELECT count(*) ,  t2.name FROM hiring AS t1 JOIN shop AS t2 ON t1.shop_id  =  t2.shop_id GROUP BY t2.name", "source_result_n": 6, "source_actual_empty": 0, "pre_screened_correct": true, "pre_screen_note": "Retained after conservative manual review of the original 50-item pilot set."}, {"item_id": "flight_2__024", "db_id": "flight_2", "question": "Which airline has abbreviation 'UAL'?", "schema": "TABLE airlines (uid INTEGER PRIMARY KEY, Airline TEXT, Abbreviation TEXT, Country TEXT);\nTABLE airports (City TEXT, AirportCode TEXT PRIMARY KEY, AirportName TEXT, Country TEXT, CountryAbbrev TEXT);\nTABLE flights (Airline INTEGER PRIMARY KEY, FlightNo INTEGER PRIMARY KEY, SourceAirport TEXT, DestAirport TEXT);\nFOREIGN KEY flights.DestAirport -> airports.AirportCode\nFOREIGN KEY flights.SourceAirport -> airports.AirportCode", "candidate_sql": "SELECT Airline FROM AIRLINES WHERE Abbreviation  =  \"UAL\"", "source_result_n": 1, "source_actual_empty": 0, "pre_screened_correct": true, "pre_screen_note": "Retained after conservative manual review of the original 50-item pilot set."}, {"item_id": "voter_1__025", "db_id": "voter_1", "question": "What are the names of the contestants whose names are not 'Jessie Alloway'", "schema": "TABLE AREA_CODE_STATE (area_code INTEGER PRIMARY KEY, state varchar(2));\nTABLE CONTESTANTS (contestant_number INTEGER PRIMARY KEY, contestant_name varchar(50));\nTABLE VOTES (vote_id INTEGER PRIMARY KEY, phone_number INTEGER, state varchar(2), contestant_number INTEGER, created timestamp);\nFOREIGN KEY VOTES.contestant_number -> CONTESTANTS.contestant_number\nFOREIGN KEY VOTES.state -> AREA_CODE_STATE.state", "candidate_sql": "SELECT contestant_name FROM contestants WHERE contestant_name != 'Jessie Alloway'", "source_result_n": 10, "source_actual_empty": 0, "pre_screened_correct": true, "pre_screen_note": "Retained after conservative manual review of the original 50-item pilot set."}, {"item_id": "tvshow__026", "db_id": "tvshow", "question": "List the language used least number of TV Channel. List language and number of TV Channel.", "schema": "TABLE Cartoon (id REAL PRIMARY KEY, Title TEXT, Directed_by TEXT, Written_by TEXT, Original_air_date TEXT, Production_code REAL, Channel TEXT);\nFOREIGN KEY Cartoon.Channel -> TV_Channel.id\nTABLE TV_Channel (id TEXT PRIMARY KEY, series_name TEXT, Country TEXT, Language TEXT, Content TEXT, Pixel_aspect_ratio_PAR TEXT, Hight_definition_TV TEXT, Pay_per_view_PPV TEXT, Package_Option TEXT);\nTABLE TV_series (id REAL PRIMARY KEY, Episode TEXT, Air_Date TEXT, Rating TEXT, Share REAL, 18_49_Rating_Share TEXT, Viewers_m TEXT, Weekly_Rank REAL, Channel TEXT);\nFOREIGN KEY TV_series.Channel -> TV_Channel.id", "candidate_sql": "SELECT LANGUAGE ,  count(*) FROM TV_Channel GROUP BY LANGUAGE ORDER BY count(*) ASC LIMIT 1;", "source_result_n": 1, "source_actual_empty": 0, "pre_screened_correct": true, "pre_screen_note": "Retained after conservative manual review of the original 50-item pilot set."}, {"item_id": "poker_player__027", "db_id": "poker_player", "question": "List the names of people that are not poker players.", "schema": "TABLE people (People_ID INT PRIMARY KEY, Nationality TEXT, Name TEXT, Birth_Date TEXT, Height REAL);\nTABLE poker_player (Poker_Player_ID INT PRIMARY KEY, People_ID INT, Final_Table_Made REAL, Best_Finish REAL, Money_Rank REAL, Earnings REAL);\nFOREIGN KEY poker_player.People_ID -> people.People_ID", "candidate_sql": "SELECT Name FROM people WHERE People_ID NOT IN (SELECT People_ID FROM poker_player)", "source_result_n": 2, "source_actual_empty": 0, "pre_screened_correct": true, "pre_screen_note": "Retained after conservative manual review of the original 50-item pilot set."}, {"item_id": "student_transcripts_tracking__028", "db_id": "student_transcripts_tracking", "question": "What is the description for the section named h?", "schema": "TABLE Addresses (address_id INTEGER PRIMARY KEY, line_1 VARCHAR(255), line_2 VARCHAR(255), line_3 VARCHAR(255), city VARCHAR(255), zip_postcode VARCHAR(20), state_province_county VARCHAR(255), country VARCHAR(255), other_address_details VARCHAR(255));\nTABLE Courses (course_id INTEGER PRIMARY KEY, course_name VARCHAR(255), course_description VARCHAR(255), other_details VARCHAR(255));\nTABLE Degree_Programs (degree_program_id INTEGER PRIMARY KEY, department_id INTEGER, degree_summary_name VARCHAR(255), degree_summary_description VARCHAR(255), other_details VARCHAR(255));\nFOREIGN KEY Degree_Programs.department_id -> Departments.department_id\nTABLE Departments (department_id INTEGER PRIMARY KEY, department_name VARCHAR(255), department_description VARCHAR(255), other_details VARCHAR(255));\nTABLE Sections (section_id INTEGER PRIMARY KEY, course_id INTEGER, section_name VARCHAR(255), section_description VARCHAR(255), other_details VARCHAR(255));\nFOREIGN KEY Sections.course_id -> Courses.course_id\nTABLE Semesters (semester_id INTEGER PRIMARY KEY, semester_name VARCHAR(255), semester_description VARCHAR(255), other_details VARCHAR(255));\nTABLE Student_Enrolment (student_enrolment_id INTEGER PRIMARY KEY, degree_program_id INTEGER, semester_id INTEGER, student_id INTEGER, other_details VARCHAR(255));\nFOREIGN KEY Student_Enrolment.student_id -> Students.student_id\nFOREIGN KEY Student_Enrolment.semester_id -> Semesters.semester_id\nFOREIGN KEY Student_Enrolment.degree_program_id -> Degree_Programs.degree_program_id\nTABLE Student_Enrolment_Courses (student_course_id INTEGER PRIMARY KEY, course_id INTEGER, student_enrolment_id INTEGER);\nFOREIGN KEY Student_Enrolment_Courses.student_enrolment_id -> Student_Enrolment.student_enrolment_id\nFOREIGN KEY Student_Enrolment_Courses.course_id -> Courses.course_id\nTABLE Students (student_id INTEGER PRIMARY KEY, current_address_id INTEGER, permanent_address_id INTEGER, first_name VARCHAR(80), middle_name VARCHAR(40), last_name VARCHAR(40), cell_mobile_number VARCHAR(40), email_address VARCHAR(40), ssn VARCHAR(40), date_first_registered DATETIME, date_left DATETIME, other_student_details VARCHAR(255));\nFOREIGN KEY Students.permanent_address_id -> Addresses.address_id\nFOREIGN KEY Students.current_address_id -> Addresses.address_id\nTABLE Transcript_Contents (student_course_id INTEGER, transcript_id INTEGER);\nFOREIGN KEY Transcript_Contents.transcript_id -> Transcripts.transcript_id\nFOREIGN KEY Transcript_Contents.student_course_id -> Student_Enrolment_Courses.student_course_id\nTABLE Transcripts (transcript_id INTEGER PRIMARY KEY, transcript_date DATETIME, other_details VARCHAR(255));", "candidate_sql": "SELECT section_description FROM Sections WHERE section_name  =  'h'", "source_result_n": 1, "source_actual_empty": 0, "pre_screened_correct": true, "pre_screen_note": "Retained after conservative manual review of the original 50-item pilot set."}, {"item_id": "course_teach__030", "db_id": "course_teach", "question": "List the most common hometown of teachers.", "schema": "TABLE course (Course_ID INT PRIMARY KEY, Staring_Date TEXT, Course TEXT);\nTABLE course_arrange (Course_ID INT PRIMARY KEY, Teacher_ID INT PRIMARY KEY, Grade INT PRIMARY KEY);\nFOREIGN KEY course_arrange.Teacher_ID -> teacher.Teacher_ID\nFOREIGN KEY course_arrange.Course_ID -> course.Course_ID\nTABLE teacher (Teacher_ID INT PRIMARY KEY, Name TEXT, Age TEXT, Hometown TEXT);", "candidate_sql": "SELECT Hometown FROM teacher GROUP BY Hometown ORDER BY COUNT(*) DESC LIMIT 1", "source_result_n": 1, "source_actual_empty": 0, "pre_screened_correct": true, "pre_screen_note": "Retained after conservative manual review of the original 50-item pilot set."}, {"item_id": "world_1__031", "db_id": "world_1", "question": "Give the name of the nation that uses the greatest amount of languages.", "schema": "TABLE city (ID INTEGER PRIMARY KEY, Name char(35), CountryCode char(3), District char(20), Population INTEGER);\nFOREIGN KEY city.CountryCode -> country.Code\nTABLE country (Code char(3) PRIMARY KEY, Name char(52), Continent TEXT, Region char(26), SurfaceArea float(10,2), IndepYear INTEGER, Population INTEGER, LifeExpectancy float(3,1), GNP float(10,2), GNPOld float(10,2), LocalName char(45), GovernmentForm char(45), HeadOfState char(60), Capital INTEGER, Code2 char(2));\nTABLE countrylanguage (CountryCode char(3) PRIMARY KEY, Language char(30) PRIMARY KEY, IsOfficial TEXT, Percentage float(4,1));\nFOREIGN KEY countrylanguage.CountryCode -> country.Code", "candidate_sql": "SELECT T1.Name FROM country AS T1 JOIN countrylanguage AS T2 ON T1.Code  =  T2.CountryCode GROUP BY T1.Name ORDER BY COUNT(*) DESC LIMIT 1", "source_result_n": 1, "source_actual_empty": 0, "pre_screened_correct": true, "pre_screen_note": "Retained after conservative manual review of the original 50-item pilot set."}, {"item_id": "cre_Doc_Template_Mgt__032", "db_id": "cre_Doc_Template_Mgt", "question": "Return the id and name of the document with the most paragraphs.", "schema": "TABLE Documents (Document_ID INTEGER PRIMARY KEY, Template_ID INTEGER, Document_Name VARCHAR(255), Document_Description VARCHAR(255), Other_Details VARCHAR(255));\nFOREIGN KEY Documents.Template_ID -> Templates.Template_ID\nTABLE Paragraphs (Paragraph_ID INTEGER PRIMARY KEY, Document_ID INTEGER, Paragraph_Text VARCHAR(255), Other_Details VARCHAR(255));\nFOREIGN KEY Paragraphs.Document_ID -> Documents.Document_ID\nTABLE Ref_Template_Types (Template_Type_Code CHAR(15) PRIMARY KEY, Template_Type_Description VARCHAR(255));\nTABLE Templates (Template_ID INTEGER PRIMARY KEY, Version_Number INTEGER, Template_Type_Code CHAR(15), Date_Effective_From DATETIME, Date_Effective_To DATETIME, Template_Details VARCHAR(255));\nFOREIGN KEY Templates.Template_Type_Code -> Ref_Template_Types.Template_Type_Code", "candidate_sql": "SELECT T1.document_id ,  T2.document_name FROM Paragraphs AS T1 JOIN Documents AS T2 ON T1.document_id  =  T2.document_id GROUP BY T1.document_id ORDER BY count(*) DESC LIMIT 1", "source_result_n": 1, "source_actual_empty": 0, "pre_screened_correct": true, "pre_screen_note": "Retained after conservative manual review of the original 50-item pilot set."}, {"item_id": "flight_2__033", "db_id": "flight_2", "question": "Which airline has most number of flights?", "schema": "TABLE airlines (uid INTEGER PRIMARY KEY, Airline TEXT, Abbreviation TEXT, Country TEXT);\nTABLE airports (City TEXT, AirportCode TEXT PRIMARY KEY, AirportName TEXT, Country TEXT, CountryAbbrev TEXT);\nTABLE flights (Airline INTEGER PRIMARY KEY, FlightNo INTEGER PRIMARY KEY, SourceAirport TEXT, DestAirport TEXT);\nFOREIGN KEY flights.DestAirport -> airports.AirportCode\nFOREIGN KEY flights.SourceAirport -> airports.AirportCode", "candidate_sql": "SELECT T1.Airline FROM AIRLINES AS T1 JOIN FLIGHTS AS T2 ON T1.uid  =  T2.Airline GROUP BY T1.Airline ORDER BY count(*) DESC LIMIT 1", "source_result_n": 1, "source_actual_empty": 0, "pre_screened_correct": true, "pre_screen_note": "Retained after conservative manual review of the original 50-item pilot set."}, {"item_id": "world_1__034", "db_id": "world_1", "question": "Which continent is Anguilla in?", "schema": "TABLE city (ID INTEGER PRIMARY KEY, Name char(35), CountryCode char(3), District char(20), Population INTEGER);\nFOREIGN KEY city.CountryCode -> country.Code\nTABLE country (Code char(3) PRIMARY KEY, Name char(52), Continent TEXT, Region char(26), SurfaceArea float(10,2), IndepYear INTEGER, Population INTEGER, LifeExpectancy float(3,1), GNP float(10,2), GNPOld float(10,2), LocalName char(45), GovernmentForm char(45), HeadOfState char(60), Capital INTEGER, Code2 char(2));\nTABLE countrylanguage (CountryCode char(3) PRIMARY KEY, Language char(30) PRIMARY KEY, IsOfficial TEXT, Percentage float(4,1));\nFOREIGN KEY countrylanguage.CountryCode -> country.Code", "candidate_sql": "SELECT Continent FROM country WHERE Name  =  \"Anguilla\"", "source_result_n": 1, "source_actual_empty": 0, "pre_screened_correct": true, "pre_screen_note": "Retained after conservative manual review of the original 50-item pilot set."}, {"item_id": "dog_kennels__035", "db_id": "dog_kennels", "question": "List each owner's first name, last name, and the size of his for her dog.", "schema": "TABLE Breeds (breed_code VARCHAR(10) PRIMARY KEY, breed_name VARCHAR(80));\nTABLE Charges (charge_id INTEGER PRIMARY KEY, charge_type VARCHAR(10), charge_amount DECIMAL(19,4));\nTABLE Dogs (dog_id INTEGER PRIMARY KEY, owner_id INTEGER, abandoned_yn VARCHAR(1), breed_code VARCHAR(10), size_code VARCHAR(10), name VARCHAR(50), age VARCHAR(20), date_of_birth DATETIME, gender VARCHAR(1), weight VARCHAR(20), date_arrived DATETIME, date_adopted DATETIME, date_departed DATETIME);\nFOREIGN KEY Dogs.owner_id -> Owners.owner_id\nFOREIGN KEY Dogs.owner_id -> Owners.owner_id\nFOREIGN KEY Dogs.size_code -> Sizes.size_code\nFOREIGN KEY Dogs.breed_code -> Breeds.breed_code\nTABLE Owners (owner_id INTEGER PRIMARY KEY, first_name VARCHAR(50), last_name VARCHAR(50), street VARCHAR(50), city VARCHAR(50), state VARCHAR(20), zip_code VARCHAR(20), email_address VARCHAR(50), home_phone VARCHAR(20), cell_number VARCHAR(20));\nTABLE Professionals (professional_id INTEGER PRIMARY KEY, role_code VARCHAR(10), first_name VARCHAR(50), street VARCHAR(50), city VARCHAR(50), state VARCHAR(20), zip_code VARCHAR(20), last_name VARCHAR(50), email_address VARCHAR(50), home_phone VARCHAR(20), cell_number VARCHAR(20));\nTABLE Sizes (size_code VARCHAR(10) PRIMARY KEY, size_description VARCHAR(80));\nTABLE Treatment_Types (treatment_type_code VARCHAR(10) PRIMARY KEY, treatment_type_description VARCHAR(80));\nTABLE Treatments (treatment_id INTEGER PRIMARY KEY, dog_id INTEGER, professional_id INTEGER, treatment_type_code VARCHAR(10), date_of_treatment DATETIME, cost_of_treatment DECIMAL(19,4));\nFOREIGN KEY Treatments.dog_id -> Dogs.dog_id\nFOREIGN KEY Treatments.professional_id -> Professionals.professional_id\nFOREIGN KEY Treatments.treatment_type_code -> Treatment_Types.treatment_type_code", "candidate_sql": "SELECT T1.first_name ,  T1.last_name ,  T2.size_code FROM Owners AS T1 JOIN Dogs AS T2 ON T1.owner_id  =  T2.owner_id", "source_result_n": 15, "source_actual_empty": 0, "pre_screened_correct": true, "pre_screen_note": "Retained after conservative manual review of the original 50-item pilot set."}, {"item_id": "student_transcripts_tracking__036", "db_id": "student_transcripts_tracking", "question": "What are the names and id of courses having at most 2 sections?", "schema": "TABLE Addresses (address_id INTEGER PRIMARY KEY, line_1 VARCHAR(255), line_2 VARCHAR(255), line_3 VARCHAR(255), city VARCHAR(255), zip_postcode VARCHAR(20), state_province_county VARCHAR(255), country VARCHAR(255), other_address_details VARCHAR(255));\nTABLE Courses (course_id INTEGER PRIMARY KEY, course_name VARCHAR(255), course_description VARCHAR(255), other_details VARCHAR(255));\nTABLE Degree_Programs (degree_program_id INTEGER PRIMARY KEY, department_id INTEGER, degree_summary_name VARCHAR(255), degree_summary_description VARCHAR(255), other_details VARCHAR(255));\nFOREIGN KEY Degree_Programs.department_id -> Departments.department_id\nTABLE Departments (department_id INTEGER PRIMARY KEY, department_name VARCHAR(255), department_description VARCHAR(255), other_details VARCHAR(255));\nTABLE Sections (section_id INTEGER PRIMARY KEY, course_id INTEGER, section_name VARCHAR(255), section_description VARCHAR(255), other_details VARCHAR(255));\nFOREIGN KEY Sections.course_id -> Courses.course_id\nTABLE Semesters (semester_id INTEGER PRIMARY KEY, semester_name VARCHAR(255), semester_description VARCHAR(255), other_details VARCHAR(255));\nTABLE Student_Enrolment (student_enrolment_id INTEGER PRIMARY KEY, degree_program_id INTEGER, semester_id INTEGER, student_id INTEGER, other_details VARCHAR(255));\nFOREIGN KEY Student_Enrolment.student_id -> Students.student_id\nFOREIGN KEY Student_Enrolment.semester_id -> Semesters.semester_id\nFOREIGN KEY Student_Enrolment.degree_program_id -> Degree_Programs.degree_program_id\nTABLE Student_Enrolment_Courses (student_course_id INTEGER PRIMARY KEY, course_id INTEGER, student_enrolment_id INTEGER);\nFOREIGN KEY Student_Enrolment_Courses.student_enrolment_id -> Student_Enrolment.student_enrolment_id\nFOREIGN KEY Student_Enrolment_Courses.course_id -> Courses.course_id\nTABLE Students (student_id INTEGER PRIMARY KEY, current_address_id INTEGER, permanent_address_id INTEGER, first_name VARCHAR(80), middle_name VARCHAR(40), last_name VARCHAR(40), cell_mobile_number VARCHAR(40), email_address VARCHAR(40), ssn VARCHAR(40), date_first_registered DATETIME, date_left DATETIME, other_student_details VARCHAR(255));\nFOREIGN KEY Students.permanent_address_id -> Addresses.address_id\nFOREIGN KEY Students.current_address_id -> Addresses.address_id\nTABLE Transcript_Contents (student_course_id INTEGER, transcript_id INTEGER);\nFOREIGN KEY Transcript_Contents.transcript_id -> Transcripts.transcript_id\nFOREIGN KEY Transcript_Contents.student_course_id -> Student_Enrolment_Courses.student_course_id\nTABLE Transcripts (transcript_id INTEGER PRIMARY KEY, transcript_date DATETIME, other_details VARCHAR(255));", "candidate_sql": "SELECT T1.course_name ,  T1.course_id FROM Courses AS T1 JOIN Sections AS T2 ON T1.course_id  =  T2.course_id GROUP BY T1.course_id HAVING count(*)  <=  2", "source_result_n": 11, "source_actual_empty": 0, "pre_screened_correct": true, "pre_screen_note": "Retained after conservative manual review of the original 50-item pilot set."}, {"item_id": "cre_Doc_Template_Mgt__037", "db_id": "cre_Doc_Template_Mgt", "question": "Show all template type codes with less than three templates.", "schema": "TABLE Documents (Document_ID INTEGER PRIMARY KEY, Template_ID INTEGER, Document_Name VARCHAR(255), Document_Description VARCHAR(255), Other_Details VARCHAR(255));\nFOREIGN KEY Documents.Template_ID -> Templates.Template_ID\nTABLE Paragraphs (Paragraph_ID INTEGER PRIMARY KEY, Document_ID INTEGER, Paragraph_Text VARCHAR(255), Other_Details VARCHAR(255));\nFOREIGN KEY Paragraphs.Document_ID -> Documents.Document_ID\nTABLE Ref_Template_Types (Template_Type_Code CHAR(15) PRIMARY KEY, Template_Type_Description VARCHAR(255));\nTABLE Templates (Template_ID INTEGER PRIMARY KEY, Version_Number INTEGER, Template_Type_Code CHAR(15), Date_Effective_From DATETIME, Date_Effective_To DATETIME, Template_Details VARCHAR(255));\nFOREIGN KEY Templates.Template_Type_Code -> Ref_Template_Types.Template_Type_Code", "candidate_sql": "SELECT template_type_code FROM Templates GROUP BY template_type_code HAVING count(*)  <  3", "source_result_n": 1, "source_actual_empty": 0, "pre_screened_correct": true, "pre_screen_note": "Retained after conservative manual review of the original 50-item pilot set."}, {"item_id": "network_1__038", "db_id": "network_1", "question": "Show the names of high schoolers who have likes, and numbers of likes for each.", "schema": "TABLE Friend (student_id INT PRIMARY KEY, friend_id INT PRIMARY KEY);\nFOREIGN KEY Friend.friend_id -> Highschooler.ID\nFOREIGN KEY Friend.student_id -> Highschooler.ID\nTABLE Highschooler (ID INT PRIMARY KEY, name TEXT, grade INT);\nTABLE Likes (student_id INT PRIMARY KEY, liked_id INT PRIMARY KEY);\nFOREIGN KEY Likes.student_id -> Highschooler.ID\nFOREIGN KEY Likes.liked_id -> Highschooler.ID", "candidate_sql": "SELECT T2.name ,  count(*) FROM Likes AS T1 JOIN Highschooler AS T2 ON T1.student_id  =  T2.id GROUP BY T1.student_id", "source_result_n": 10, "source_actual_empty": 0, "pre_screened_correct": true, "pre_screen_note": "Retained after conservative manual review of the original 50-item pilot set."}, {"item_id": "course_teach__040", "db_id": "course_teach", "question": "Show the hometowns shared by at least two teachers.", "schema": "TABLE course (Course_ID INT PRIMARY KEY, Staring_Date TEXT, Course TEXT);\nTABLE course_arrange (Course_ID INT PRIMARY KEY, Teacher_ID INT PRIMARY KEY, Grade INT PRIMARY KEY);\nFOREIGN KEY course_arrange.Teacher_ID -> teacher.Teacher_ID\nFOREIGN KEY course_arrange.Course_ID -> course.Course_ID\nTABLE teacher (Teacher_ID INT PRIMARY KEY, Name TEXT, Age TEXT, Hometown TEXT);", "candidate_sql": "SELECT Hometown FROM teacher GROUP BY Hometown HAVING COUNT(*)  >=  2", "source_result_n": 0, "source_actual_empty": 1, "pre_screened_correct": true, "pre_screen_note": "Retained after conservative manual review of the original 50-item pilot set."}, {"item_id": "wta_1__041", "db_id": "wta_1", "question": "How many total tours were there for each ranking date?", "schema": "TABLE matches (best_of INT, draw_size INT, loser_age FLOAT, loser_entry TEXT, loser_hand TEXT, loser_ht INT, loser_id INT, loser_ioc TEXT, loser_name TEXT, loser_rank INT, loser_rank_points INT, loser_seed INT, match_num INT, minutes INT, round TEXT, score TEXT, surface TEXT, tourney_date DATE, tourney_id TEXT, tourney_level TEXT, tourney_name TEXT, winner_age FLOAT, winner_entry TEXT, winner_hand TEXT, winner_ht INT, winner_id INT, winner_ioc TEXT, winner_name TEXT, winner_rank INT, winner_rank_points INT, winner_seed INT, year INT);\nFOREIGN KEY matches.winner_id -> players.player_id\nFOREIGN KEY matches.loser_id -> players.player_id\nTABLE players (player_id INT PRIMARY KEY, first_name TEXT, last_name TEXT, hand TEXT, birth_date DATE, country_code TEXT);\nTABLE rankings (ranking_date DATE, ranking INT, player_id INT, ranking_points INT, tours INT);\nFOREIGN KEY rankings.player_id -> players.player_id", "candidate_sql": "SELECT sum(tours) ,  ranking_date FROM rankings GROUP BY ranking_date", "source_result_n": 1504, "source_actual_empty": 0, "pre_screened_correct": true, "pre_screen_note": "Retained after conservative manual review of the original 50-item pilot set."}, {"item_id": "world_1__042", "db_id": "world_1", "question": "What are the region and population of Angola?", "schema": "TABLE city (ID INTEGER PRIMARY KEY, Name char(35), CountryCode char(3), District char(20), Population INTEGER);\nFOREIGN KEY city.CountryCode -> country.Code\nTABLE country (Code char(3) PRIMARY KEY, Name char(52), Continent TEXT, Region char(26), SurfaceArea float(10,2), IndepYear INTEGER, Population INTEGER, LifeExpectancy float(3,1), GNP float(10,2), GNPOld float(10,2), LocalName char(45), GovernmentForm char(45), HeadOfState char(60), Capital INTEGER, Code2 char(2));\nTABLE countrylanguage (CountryCode char(3) PRIMARY KEY, Language char(30) PRIMARY KEY, IsOfficial TEXT, Percentage float(4,1));\nFOREIGN KEY countrylanguage.CountryCode -> country.Code", "candidate_sql": "SELECT Population ,  Region FROM country WHERE Name  =  \"Angola\"", "source_result_n": 1, "source_actual_empty": 0, "pre_screened_correct": true, "pre_screen_note": "Retained after conservative manual review of the original 50-item pilot set."}, {"item_id": "world_1__043", "db_id": "world_1", "question": "What are the names of all the countries that became independent after 1950?", "schema": "TABLE city (ID INTEGER PRIMARY KEY, Name char(35), CountryCode char(3), District char(20), Population INTEGER);\nFOREIGN KEY city.CountryCode -> country.Code\nTABLE country (Code char(3) PRIMARY KEY, Name char(52), Continent TEXT, Region char(26), SurfaceArea float(10,2), IndepYear INTEGER, Population INTEGER, LifeExpectancy float(3,1), GNP float(10,2), GNPOld float(10,2), LocalName char(45), GovernmentForm char(45), HeadOfState char(60), Capital INTEGER, Code2 char(2));\nTABLE countrylanguage (CountryCode char(3) PRIMARY KEY, Language char(30) PRIMARY KEY, IsOfficial TEXT, Percentage float(4,1));\nFOREIGN KEY countrylanguage.CountryCode -> country.Code", "candidate_sql": "SELECT Name FROM country WHERE IndepYear  >  1950", "source_result_n": 110, "source_actual_empty": 0, "pre_screened_correct": true, "pre_screen_note": "Retained after conservative manual review of the original 50-item pilot set."}, {"item_id": "poker_player__044", "db_id": "poker_player", "question": "Return the names of poker players sorted by their earnings descending.", "schema": "TABLE people (People_ID INT PRIMARY KEY, Nationality TEXT, Name TEXT, Birth_Date TEXT, Height REAL);\nTABLE poker_player (Poker_Player_ID INT PRIMARY KEY, People_ID INT, Final_Table_Made REAL, Best_Finish REAL, Money_Rank REAL, Earnings REAL);\nFOREIGN KEY poker_player.People_ID -> people.People_ID", "candidate_sql": "SELECT T1.Name FROM people AS T1 JOIN poker_player AS T2 ON T1.People_ID  =  T2.People_ID ORDER BY T2.Earnings DESC", "source_result_n": 5, "source_actual_empty": 0, "pre_screened_correct": true, "pre_screen_note": "Retained after conservative manual review of the original 50-item pilot set."}, {"item_id": "flight_2__046", "db_id": "flight_2", "question": "What are flight numbers of flights departing from Airport \"APG\"?", "schema": "TABLE airlines (uid INTEGER PRIMARY KEY, Airline TEXT, Abbreviation TEXT, Country TEXT);\nTABLE airports (City TEXT, AirportCode TEXT PRIMARY KEY, AirportName TEXT, Country TEXT, CountryAbbrev TEXT);\nTABLE flights (Airline INTEGER PRIMARY KEY, FlightNo INTEGER PRIMARY KEY, SourceAirport TEXT, DestAirport TEXT);\nFOREIGN KEY flights.DestAirport -> airports.AirportCode\nFOREIGN KEY flights.SourceAirport -> airports.AirportCode", "candidate_sql": "SELECT FlightNo FROM FLIGHTS WHERE SourceAirport  =  \"APG\"", "source_result_n": 0, "source_actual_empty": 1, "pre_screened_correct": true, "pre_screen_note": "Retained after conservative manual review of the original 50-item pilot set."}, {"item_id": "car_1__047", "db_id": "car_1", "question": "What are the names and ids of all makers with more than 3 models?", "schema": "TABLE car_makers (Id INTEGER PRIMARY KEY, Maker TEXT, FullName TEXT, Country TEXT);\nFOREIGN KEY car_makers.Country -> countries.CountryId\nTABLE car_names (MakeId INTEGER PRIMARY KEY, Model TEXT, Make TEXT);\nFOREIGN KEY car_names.Model -> model_list.Model\nTABLE cars_data (Id INTEGER PRIMARY KEY, MPG TEXT, Cylinders INTEGER, Edispl REAL, Horsepower TEXT, Weight INTEGER, Accelerate REAL, Year INTEGER);\nFOREIGN KEY cars_data.Id -> car_names.MakeId\nTABLE continents (ContId INTEGER PRIMARY KEY, Continent TEXT);\nTABLE countries (CountryId INTEGER PRIMARY KEY, CountryName TEXT, Continent INTEGER);\nFOREIGN KEY countries.Continent -> continents.ContId\nTABLE model_list (ModelId INTEGER PRIMARY KEY, Maker INTEGER, Model TEXT);\nFOREIGN KEY model_list.Maker -> car_makers.Id", "candidate_sql": "SELECT T1.FullName ,  T1.Id FROM CAR_MAKERS AS T1 JOIN MODEL_LIST AS T2 ON T1.Id  =  T2.Maker GROUP BY T1.Id HAVING count(*)  >  3;", "source_result_n": 2, "source_actual_empty": 0, "pre_screened_correct": true, "pre_screen_note": "Retained after conservative manual review of the original 50-item pilot set."}, {"item_id": "voter_1__048", "db_id": "voter_1", "question": "What are the contestant numbers and names of the contestants who had at least two votes?", "schema": "TABLE AREA_CODE_STATE (area_code INTEGER PRIMARY KEY, state varchar(2));\nTABLE CONTESTANTS (contestant_number INTEGER PRIMARY KEY, contestant_name varchar(50));\nTABLE VOTES (vote_id INTEGER PRIMARY KEY, phone_number INTEGER, state varchar(2), contestant_number INTEGER, created timestamp);\nFOREIGN KEY VOTES.contestant_number -> CONTESTANTS.contestant_number\nFOREIGN KEY VOTES.state -> AREA_CODE_STATE.state", "candidate_sql": "SELECT T1.contestant_number , T1.contestant_name FROM contestants AS T1 JOIN votes AS T2 ON T1.contestant_number  =  T2.contestant_number GROUP BY T1.contestant_number HAVING count(*)  >=  2", "source_result_n": 1, "source_actual_empty": 0, "pre_screened_correct": true, "pre_screen_note": "Retained after conservative manual review of the original 50-item pilot set."}]"""

ITEMS = pd.DataFrame(
    json.loads(CURATED_DATA_JSON)
)

# Stable IDs for the new experiment.
ITEMS["confirmation_id"] = [
    f"CONF_{i:03d}"
    for i in range(len(ITEMS))
]

# Dataset QA.
assert len(ITEMS) == 40
assert ITEMS["confirmation_id"].is_unique
assert not ITEMS["question"].duplicated().any()
assert not ITEMS["candidate_sql"].duplicated().any()
assert ITEMS["pre_screened_correct"].eq(True).all()
assert ITEMS["source_actual_empty"].isin([0, 1, False, True]).all()

print("Embedded items:", len(ITEMS))
print("Databases:", ITEMS["db_id"].nunique())
print("Source actual-empty items:", int(ITEMS["source_actual_empty"].sum()))

display(
    ITEMS[[
        "confirmation_id",
        "item_id",
        "db_id",
        "question",
        "source_result_n",
        "source_actual_empty",
    ]].head(10)
)

In [ ]:
# ============================================================
# 4. TEAM AUDIT SHEET
# ============================================================
AUDIT_PATH = WORK / "pilot_manual_audit.csv"

if AUDIT_PATH.exists():
    AUDIT = pd.read_csv(AUDIT_PATH)
    print("Loaded existing audit sheet:", AUDIT_PATH)
else:
    AUDIT = ITEMS[[
        "confirmation_id",
        "item_id",
        "db_id",
        "question",
        "candidate_sql",
        "schema",
        "source_result_n",
        "source_actual_empty",
        "pre_screened_correct",
        "pre_screen_note",
    ]].copy()

    AUDIT["team_question_clear"] = ""
    AUDIT["team_candidate_correct"] = ""
    AUDIT["team_include"] = ""
    AUDIT["team_notes"] = ""

    AUDIT.to_csv(
        AUDIT_PATH,
        index=False
    )

    print("Created audit sheet:", AUDIT_PATH)

def is_yes(x):
    return str(x).strip().lower() in {
        "yes", "y", "1", "true", "ok", "include"
    }

if CFG["require_team_audit"]:
    required_cols = {
        "team_question_clear",
        "team_candidate_correct",
        "team_include",
    }

    if not required_cols.issubset(AUDIT.columns):
        raise RuntimeError(
            "Audit CSV is missing required team-audit columns."
        )

    keep_ids = set(
        AUDIT[
            AUDIT["team_question_clear"].map(is_yes)
            & AUDIT["team_candidate_correct"].map(is_yes)
            & AUDIT["team_include"].map(is_yes)
        ]["confirmation_id"].astype(str)
    )

    ITEMS = ITEMS[
        ITEMS["confirmation_id"].isin(keep_ids)
    ].copy()

    if len(ITEMS) < 20:
        raise RuntimeError(
            f"Only {len(ITEMS)} team-audited items remain. "
            "Audit more rows before running the confirmation experiment."
        )

    print("Using team-audited items:", len(ITEMS))
else:
    print(
        "Using the embedded conservative pre-screen set. "
        "For paper-quality confirmation, fill pilot_manual_audit.csv "
        "and set CFG['require_team_audit']=True."
    )

In [ ]:
# ============================================================
# 5. FOUR FEEDBACK CONDITIONS
# ============================================================
CONDITION_TEXT = {
    "NONE": "",
    "SUCCESS_ONLY": (
        "Execution observation:\n"
        "Status: SUCCESS."
    ),
    "EMPTY": (
        "Execution observation:\n"
        "Status: SUCCESS.\n"
        "Returned row count: 0."
    ),
    "NONEMPTY": (
        "Execution observation:\n"
        "Status: SUCCESS.\n"
        "Returned row count: 1."
    ),
}

# Sanity check: EMPTY and SUCCESS_ONLY differ only by the row-count line.
assert CONDITION_TEXT["EMPTY"].startswith(
    CONDITION_TEXT["SUCCESS_ONLY"]
)
assert CONDITION_TEXT["NONEMPTY"].startswith(
    CONDITION_TEXT["SUCCESS_ONLY"]
)

print(json.dumps(CONDITION_TEXT, indent=2))

In [ ]:
# ============================================================
# 6. EXPAND PAIRED TRIALS
# ============================================================
trial_rows = []

for _, row in ITEMS.iterrows():
    for condition, feedback_text in CONDITION_TEXT.items():
        trial_rows.append({
            **row.to_dict(),
            "condition": condition,
            "feedback_text": feedback_text,
        })

TRIALS = pd.DataFrame(trial_rows)

if CFG["shuffle_trials"]:
    TRIALS = TRIALS.sample(
        frac=1.0,
        random_state=CFG["seed"],
    ).reset_index(drop=True)

print("Items:", len(ITEMS))
print("Trials/model:", len(TRIALS))

display(
    TRIALS.groupby("condition")
    .size()
    .to_frame("n")
)

TRIALS.to_csv(
    WORK / "pilot_trials.csv",
    index=False
)

In [ ]:
# ============================================================
# 7. VERDICT-ONLY PROMPT
# ============================================================
SYSTEM_PROMPT = """You are a careful SQLite semantic verifier.
Determine whether the candidate SQL correctly answers the natural-language question for the given database schema.
An execution observation may or may not be provided.

Return exactly one of these two JSON objects:
{"verdict":"CORRECT"}
{"verdict":"INCORRECT"}
"""

def make_user_prompt(row):
    feedback_block = ""

    if row["condition"] != "NONE":
        feedback_block = (
            "\n\n"
            + row["feedback_text"]
        )

    return f"""Natural-language question:
{row['question']}

SQLite schema:
{row['schema']}

Candidate SQL:
{row['candidate_sql']}{feedback_block}

Does the candidate SQL correctly answer the question?
Return JSON only."""

print(make_user_prompt(TRIALS.iloc[0])[:3000])

In [ ]:
# ============================================================
# 8. OUTPUT PARSER + SMOKE TESTS
# ============================================================
def parse_verdict(text):
    raw = str(text or "").strip()

    # Strict JSON first.
    try:
        obj = json.loads(raw)

        if isinstance(obj, dict):
            verdict = str(
                obj.get("verdict", "")
            ).strip().upper()

            if verdict in {"CORRECT", "INCORRECT"}:
                return {
                    "parsed_verdict": verdict,
                    "parse_ok": True,
                    "strict_json": True,
                }
    except Exception:
        pass

    # Embedded JSON object.
    for blob in re.findall(
        r"\{.*?\}",
        raw,
        flags=re.S,
    ):
        try:
            obj = json.loads(blob)

            verdict = str(
                obj.get("verdict", "")
            ).strip().upper()

            if verdict in {"CORRECT", "INCORRECT"}:
                return {
                    "parsed_verdict": verdict,
                    "parse_ok": True,
                    "strict_json": False,
                }
        except Exception:
            pass

    # Conservative fallback.
    upper = raw.upper()

    if re.search(r"\bINCORRECT\b", upper):
        return {
            "parsed_verdict": "INCORRECT",
            "parse_ok": True,
            "strict_json": False,
        }

    if re.search(r"\bCORRECT\b", upper):
        return {
            "parsed_verdict": "CORRECT",
            "parse_ok": True,
            "strict_json": False,
        }

    return {
        "parsed_verdict": "",
        "parse_ok": False,
        "strict_json": False,
    }

assert parse_verdict('{"verdict":"CORRECT"}')["parsed_verdict"] == "CORRECT"
assert parse_verdict('{"verdict":"INCORRECT"}')["parsed_verdict"] == "INCORRECT"
assert parse_verdict("INCORRECT")["parsed_verdict"] == "INCORRECT"
assert parse_verdict("CORRECT")["parsed_verdict"] == "CORRECT"

print("✓ Verdict parser smoke tests passed.")

In [ ]:
# ============================================================
# 9. MODEL ADAPTER
# ============================================================
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

def preferred_dtype():
    if not torch.cuda.is_available():
        return torch.float32

    major, _ = torch.cuda.get_device_capability(0)

    return (
        torch.bfloat16
        if major >= 8
        else torch.float16
    )

def make_quant_config():
    if not (
        CFG["load_in_4bit"]
        and torch.cuda.is_available()
    ):
        return None

    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=preferred_dtype(),
    )

class TextModel:
    def __init__(self, model_id):
        self.model_id = model_id

        kwargs = {
            "torch_dtype": preferred_dtype(),
            "trust_remote_code": CFG["trust_remote_code"],
        }

        if torch.cuda.is_available():
            kwargs["device_map"] = "auto"

        qcfg = make_quant_config()

        if qcfg is not None:
            kwargs["quantization_config"] = qcfg

        if HF_TOKEN:
            kwargs["token"] = HF_TOKEN

        self.tokenizer = AutoTokenizer.from_pretrained(
            model_id,
            token=HF_TOKEN if HF_TOKEN else None,
            trust_remote_code=CFG["trust_remote_code"],
        )

        self.model = AutoModelForCausalLM.from_pretrained(
            model_id,
            **kwargs
        ).eval()

    @property
    def input_device(self):
        try:
            return self.model.get_input_embeddings().weight.device
        except Exception:
            return next(self.model.parameters()).device

    def generate(self, system_prompt, user_prompt):
        messages = [
            {
                "role": "system",
                "content": system_prompt,
            },
            {
                "role": "user",
                "content": user_prompt,
            },
        ]

        template_kwargs = {
            "add_generation_prompt": True,
            "tokenize": True,
            "return_dict": True,
            "return_tensors": "pt",
        }

        if "qwen3" in self.model_id.lower():
            template_kwargs["enable_thinking"] = False

        try:
            inputs = self.tokenizer.apply_chat_template(
                messages,
                **template_kwargs
            )
        except TypeError:
            template_kwargs.pop(
                "enable_thinking",
                None
            )

            inputs = self.tokenizer.apply_chat_template(
                messages,
                **template_kwargs
            )

        inputs = {
            k: v.to(self.input_device)
            for k, v in inputs.items()
        }

        input_len = inputs["input_ids"].shape[-1]

        generate_kwargs = {
            "max_new_tokens": CFG["max_new_tokens"],
            "do_sample": CFG["temperature"] > 0,
            "pad_token_id": self.tokenizer.eos_token_id,
        }

        if CFG["temperature"] > 0:
            generate_kwargs["temperature"] = CFG["temperature"]

        with torch.inference_mode():
            out = self.model.generate(
                **inputs,
                **generate_kwargs
            )

        return self.tokenizer.decode(
            out[0][input_len:],
            skip_special_tokens=True
        ).strip()

    def close(self):
        try:
            del self.model
            del self.tokenizer
        except Exception:
            pass

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

In [ ]:
# ============================================================
# 10. RUN — AUTOSAVE + RESUME
# ============================================================
RESULT_PATH = WORK / "pilot_model_results.csv"

if (
    RESULT_PATH.exists()
    and not CFG["overwrite_results"]
):
    RESULTS = pd.read_csv(RESULT_PATH)
    print("Resuming:", len(RESULTS), "saved rows")
else:
    RESULTS = pd.DataFrame()

def completed_keys(df):
    if df.empty:
        return set()

    return set(zip(
        df["model_id"].astype(str),
        df["confirmation_id"].astype(str),
        df["condition"].astype(str),
    ))

for model_id in CFG["models"]:
    done = completed_keys(RESULTS)

    remaining = TRIALS[
        ~TRIALS.apply(
            lambda r: (
                model_id,
                r["confirmation_id"],
                r["condition"],
            ) in done,
            axis=1,
        )
    ]

    if remaining.empty:
        print("Already complete:", model_id)
        continue

    print("\n" + "=" * 88)
    print("Loading:", model_id)
    print("Remaining calls:", len(remaining))

    backend = TextModel(model_id)
    buffer = []
    early_latencies = []

    try:
        for step, (_, row) in enumerate(
            tqdm(
                remaining.iterrows(),
                total=len(remaining),
                desc=model_id,
            ),
            start=1,
        ):
            started = time.time()
            error = None

            try:
                raw = backend.generate(
                    SYSTEM_PROMPT,
                    make_user_prompt(row),
                )
            except Exception as e:
                raw = ""
                error = (
                    f"{type(e).__name__}: {e}"
                )

            latency = time.time() - started

            if len(early_latencies) < 20:
                early_latencies.append(latency)

            parsed = parse_verdict(raw)

            buffer.append({
                "model_id": model_id,
                "confirmation_id": row["confirmation_id"],
                "item_id": row["item_id"],
                "db_id": row["db_id"],
                "condition": row["condition"],
                "question": row["question"],
                "candidate_sql": row["candidate_sql"],
                "source_result_n": row["source_result_n"],
                "source_actual_empty": row["source_actual_empty"],
                "raw_output": raw,
                "latency_s": latency,
                "error": error,
                **parsed,
            })

            if len(buffer) >= 10:
                RESULTS = pd.concat(
                    [
                        RESULTS,
                        pd.DataFrame(buffer),
                    ],
                    ignore_index=True,
                )

                RESULTS.to_csv(
                    RESULT_PATH,
                    index=False
                )

                buffer = []

            if step == 10 and early_latencies:
                avg_s = float(
                    np.mean(early_latencies)
                )

                eta_min = (
                    (len(remaining) - step)
                    * avg_s
                    / 60
                )

                print(
                    f"\nEarly ETA for remaining calls: "
                    f"{eta_min:.1f} min "
                    f"({avg_s:.2f}s/call)"
                )

        if buffer:
            RESULTS = pd.concat(
                [
                    RESULTS,
                    pd.DataFrame(buffer),
                ],
                ignore_index=True,
            )

            RESULTS.to_csv(
                RESULT_PATH,
                index=False
            )

    finally:
        backend.close()

print("\nSaved result rows:", len(RESULTS))
print(RESULT_PATH)

In [ ]:
# ============================================================
# 11. PRIMARY PAIRED ANALYSIS
# ============================================================
R = RESULTS[
    RESULTS["parse_ok"] == True
].copy()

R["reject_correct"] = (
    R["parsed_verdict"] == "INCORRECT"
).astype(int)

R["accept_correct"] = (
    R["parsed_verdict"] == "CORRECT"
).astype(int)

def paired_binary_effect(
    df,
    cond_a,
    cond_b,
    outcome_col,
):
    piv = df.pivot_table(
        index="confirmation_id",
        columns="condition",
        values=outcome_col,
        aggfunc="first",
    )

    if (
        cond_a not in piv.columns
        or cond_b not in piv.columns
    ):
        return None

    pair = piv[
        [cond_a, cond_b]
    ].dropna()

    if len(pair) == 0:
        return None

    a = pair[cond_a].astype(int)
    b = pair[cond_b].astype(int)

    a1_b0 = int(
        ((a == 1) & (b == 0)).sum()
    )

    a0_b1 = int(
        ((a == 0) & (b == 1)).sum()
    )

    discord = a1_b0 + a0_b1

    p_value = (
        binomtest(
            a1_b0,
            discord,
            0.5,
        ).pvalue
        if discord
        else 1.0
    )

    return {
        "n_pairs": len(pair),
        "rate_a": float(a.mean()),
        "rate_b": float(b.mean()),
        "diff_a_minus_b": float(
            (a - b).mean()
        ),
        "a1_b0": a1_b0,
        "a0_b1": a0_b1,
        "discordant_pairs": discord,
        "exact_mcnemar_p": p_value,
    }

VERDICT_SUMMARY = (
    R.groupby(
        ["model_id", "condition"]
    )
    .agg(
        n_parsed=("confirmation_id", "count"),
        rejection_rate=("reject_correct", "mean"),
        acceptance_rate=("accept_correct", "mean"),
        mean_latency_s=("latency_s", "mean"),
    )
    .reset_index()
)

OUTPUT_QUALITY = (
    RESULTS.groupby(
        ["model_id", "condition"]
    )
    .agg(
        n_total=("confirmation_id", "count"),
        parse_rate=("parse_ok", "mean"),
        strict_json_rate=("strict_json", "mean"),
    )
    .reset_index()
)

SUMMARY = VERDICT_SUMMARY.merge(
    OUTPUT_QUALITY,
    on=["model_id", "condition"],
    how="outer",
)

display(SUMMARY)

SUMMARY.to_csv(
    WORK / "pilot_summary.csv",
    index=False
)

effect_rows = []

for model_id, dm in R.groupby("model_id"):
    comparisons = [
        (
            "EMPTY",
            "SUCCESS_ONLY",
            "PRIMARY: EMPTY vs SUCCESS_ONLY rejection"
        ),
        (
            "EMPTY",
            "NONEMPTY",
            "EMPTY vs NONEMPTY rejection"
        ),
        (
            "EMPTY",
            "NONE",
            "EMPTY vs NONE rejection"
        ),
        (
            "NONEMPTY",
            "SUCCESS_ONLY",
            "NONEMPTY vs SUCCESS_ONLY rejection"
        ),
        (
            "SUCCESS_ONLY",
            "NONE",
            "SUCCESS_ONLY vs NONE rejection"
        ),
    ]

    for a, b, label in comparisons:
        effect = paired_binary_effect(
            dm,
            a,
            b,
            "reject_correct",
        )

        if effect:
            effect_rows.append({
                "model_id": model_id,
                "analysis": label,
                **effect,
            })

EFFECTS = pd.DataFrame(
    effect_rows
)

display(EFFECTS)

EFFECTS.to_csv(
    WORK / "pilot_effects.csv",
    index=False
)

In [ ]:
# ============================================================
# 12. CORE GO / NO-GO READOUT
# ============================================================
def rejection_rate(df, model, condition):
    x = df[
        (df["model_id"] == model)
        & (df["condition"] == condition)
    ]

    return (
        float(x["reject_correct"].mean())
        if len(x)
        else np.nan
    )

core_rows = []

for model in sorted(
    R["model_id"].unique()
):
    none = rejection_rate(
        R, model, "NONE"
    )

    success = rejection_rate(
        R, model, "SUCCESS_ONLY"
    )

    empty = rejection_rate(
        R, model, "EMPTY"
    )

    nonempty = rejection_rate(
        R, model, "NONEMPTY"
    )

    primary_shift = (
        empty - success
    )

    core_rows.append({
        "model": model,
        "Reject correct | NONE": none,
        "Reject correct | SUCCESS_ONLY": success,
        "Reject correct | EMPTY": empty,
        "Reject correct | NONEMPTY": nonempty,
        "PRIMARY empty shift vs success-only": primary_shift,
        "Empty shift vs nonempty": empty - nonempty,
        "Success-only shift vs none": success - none,
        "Nonempty shift vs success-only": nonempty - success,
    })

CORE = pd.DataFrame(core_rows)

display(CORE)

CORE.to_csv(
    WORK / "pilot_core_readout.csv",
    index=False
)

for _, row in CORE.iterrows():
    shift_pp = (
        100
        * row[
            "PRIMARY empty shift vs success-only"
        ]
    )

    print(
        "\nMODEL:",
        row["model"]
    )

    print(
        f"Primary EMPTY vs SUCCESS_ONLY shift: "
        f"{shift_pp:+.1f} percentage points"
    )

    if shift_pp >= CFG["strong_go_pp"]:
        print(
            "→ STRONG GO signal for scaling / second-model confirmation."
        )
    elif shift_pp >= CFG["moderate_go_pp"]:
        print(
            "→ MODERATE GO: replicate with second model / more audited items."
        )
    elif shift_pp >= 5:
        print(
            "→ WEAK / BORDERLINE: inspect flips before scaling."
        )
    else:
        print(
            "→ NO-GO for the original strong empty-rejection story."
        )

In [ ]:
# ============================================================
# 13. ITEM-LEVEL FLIPS + DATABASE SPREAD
# ============================================================
transition_frames = []

for model_id, dm in R.groupby(
    "model_id"
):
    piv = dm.pivot_table(
        index=[
            "confirmation_id",
            "item_id",
            "db_id",
            "question",
            "candidate_sql",
            "source_actual_empty",
        ],
        columns="condition",
        values="reject_correct",
        aggfunc="first",
    ).reset_index()

    if {
        "EMPTY",
        "SUCCESS_ONLY",
    }.issubset(piv.columns):
        piv[
            "success_accept_to_empty_reject"
        ] = (
            (piv["SUCCESS_ONLY"] == 0)
            & (piv["EMPTY"] == 1)
        )

        piv[
            "success_reject_to_empty_accept"
        ] = (
            (piv["SUCCESS_ONLY"] == 1)
            & (piv["EMPTY"] == 0)
        )

        forward = piv[
            piv[
                "success_accept_to_empty_reject"
            ]
        ]

        reverse = piv[
            piv[
                "success_reject_to_empty_accept"
            ]
        ]

        print("\nMODEL:", model_id)
        print(
            "SUCCESS_ONLY accept → EMPTY reject:",
            len(forward)
        )
        print(
            "SUCCESS_ONLY reject → EMPTY accept:",
            len(reverse)
        )
        print(
            "DBs contributing forward flips:",
            forward["db_id"].nunique()
        )

        display(
            forward[[
                "confirmation_id",
                "db_id",
                "question",
                "source_actual_empty",
            ]].head(30)
        )

    piv.insert(
        0,
        "model_id",
        model_id
    )

    transition_frames.append(
        piv
    )

TRANSITIONS = (
    pd.concat(
        transition_frames,
        ignore_index=True
    )
    if transition_frames
    else pd.DataFrame()
)

TRANSITIONS.to_csv(
    WORK / "pilot_transitions.csv",
    index=False
)

In [ ]:
# ============================================================
# 14. EXPLORATORY: SOURCE-ACTUAL-EMPTY SUBGROUP
# ============================================================
# This is descriptive only: the subgroup is very small.
# It asks whether the same pattern is visible on candidates that
# genuinely returned zero rows in the source database.

ACTUAL_EMPTY = R[
    R["source_actual_empty"] == 1
].copy()

ACTUAL_NONEMPTY = R[
    R["source_actual_empty"] == 0
].copy()

print(
    "Unique source-actual-empty items:",
    ACTUAL_EMPTY["confirmation_id"].nunique()
)

print(
    "Unique source-nonempty items:",
    ACTUAL_NONEMPTY["confirmation_id"].nunique()
)

if ACTUAL_EMPTY["confirmation_id"].nunique() > 0:
    display(
        ACTUAL_EMPTY.groupby(
            ["model_id", "condition"]
        )["reject_correct"]
        .agg(["count", "mean"])
        .rename(
            columns={
                "mean": "rejection_rate"
            }
        )
    )

In [ ]:
import shutil

# ============================================================
# 15. PACKAGE PERSISTENT OUTPUTS
# ============================================================
archive = shutil.make_archive(
    str(
        WORK
        / "execution_outcome_confirmation_outputs"
    ),
    "zip",
    root_dir=WORK,
)

print("Persistent output folder:")
print(WORK.resolve())

print("\nArchive:")
print(archive)

print("\nKey files:")

for name in [
    "pilot_manual_audit.csv",
    "pilot_trials.csv",
    "pilot_model_results.csv",
    "pilot_summary.csv",
    "pilot_effects.csv",
    "pilot_core_readout.csv",
    "pilot_transitions.csv",
]:
    p = WORK / name

    print(
        ("✓" if p.exists() else "·"),
        p
    )

## 3. Interpretation boundaries

This confirmation re-pilot directly supports only a **reported-feedback effect**:

> Adding a zero-row cardinality message changes verifier judgments for the same correct SQL candidate.

The strongest confirmation pattern is:

```text
SUCCESS_ONLY → CORRECT
EMPTY        → INCORRECT
```

with few or no reverse flips, spread across multiple databases.

### What would strengthen the research next?

If the primary `EMPTY vs SUCCESS_ONLY` effect survives:

1. replicate on the second model family;
2. perform independent team annotation of correct SQL;
3. add authentic paired-database execution;
4. add a falsification arm where feedback is declared randomized/non-informative;
5. only then decide whether the stronger term **execution-outcome bias** is justified.

If the effect collapses after verdict-only prompting + success-only control, the first pilot likely mixed cardinality effects with debugging/task-framing effects.